# 🏗️ Model 3 — Transformer Architecture for Construction Site Digital Twin & Progress Simulation

**Project:** ConstructionSite AI — Multi-Modal Generative AI Framework  
**Objective:** Implement Transformer-based Generative AI architectures for Construction Site Digital Twin representation, Future Progress Simulation, and Spatial Risk Intelligence.

### Models Implemented:
1. **Vision Transformer Autoencoder (ViT-AE)**: Patch-based spatial self-attention autoencoder for fine-grained digital twin state representation.
2. **Construction Progress Transformer**: Multi-modal cross-attention sequence transformer that takes current site visual tokens and conditions on target BIM/Schedule stage tokens to simulate future construction states and predict spatial hazard maps.

---


## 1 · Imports & Environment Configuration


In [1]:
import os
import glob
import random
import time
import warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid')

# Dynamic Base Paths
BASE = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA = os.path.join(BASE, 'ConstructionSiteSafetyImageDatasetRoboflow', 'css-data')
OUT  = os.path.join(BASE, 'data', 'processed', 'features')
os.makedirs(OUT, exist_ok=True)

# Model Hyperparameters
IMG_SIZE     = 128
PATCH_SIZE   = 16          # 128 / 16 = 8x8 = 64 patches
NUM_PATCHES  = (IMG_SIZE // PATCH_SIZE) ** 2
EMBED_DIM    = 256
NUM_HEADS    = 8
ENC_DEPTH    = 6
DEC_DEPTH    = 4
DEC_EMBED    = 128
BATCH_SIZE   = 32
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device             : {DEVICE}')
print(f'Image Size         : {IMG_SIZE}x{IMG_SIZE} (Patches: {NUM_PATCHES} of {PATCH_SIZE}x{PATCH_SIZE})')
print(f'Transformer Dim    : {EMBED_DIM} (Heads: {NUM_HEADS}, Enc Depth: {ENC_DEPTH}, Dec Depth: {DEC_DEPTH})')


Device             : cuda
Image Size         : 128x128 (Patches: 64 of 16x16)
Transformer Dim    : 256 (Heads: 8, Enc Depth: 6, Dec Depth: 4)


## 2 · Dataset & DataLoader


In [2]:
class ConstructionImageDataset(Dataset):
    def __init__(self, img_dir, transform=None):
        self.img_paths = sorted(glob.glob(os.path.join(img_dir, '*.jpg')))
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img = Image.open(self.img_paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

train_ds = ConstructionImageDataset(os.path.join(DATA, 'train', 'images'), transform)
val_ds   = ConstructionImageDataset(os.path.join(DATA, 'valid', 'images'), transform)
test_ds  = ConstructionImageDataset(os.path.join(DATA, 'test',  'images'), transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f'Train samples: {len(train_ds)} | Valid samples: {len(val_ds)} | Test samples: {len(test_ds)}')


Train samples: 2605 | Valid samples: 114 | Test samples: 82


## 3 · Patch Embedding & Positional Encodings


In [3]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=128, patch_size=16, in_channels=3, embed_dim=256):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        B, C, H, W = x.shape
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x

sample_x = torch.randn(2, 3, 128, 128)
patch_embed_layer = PatchEmbedding(IMG_SIZE, PATCH_SIZE, 3, EMBED_DIM)
sample_patches = patch_embed_layer(sample_x)
print(f'Input shape        : {sample_x.shape}')
print(f'Patches token shape: {sample_patches.shape} (N={sample_patches.shape[1]} tokens, D={sample_patches.shape[2]})')


Input shape        : torch.Size([2, 3, 128, 128])
Patches token shape: torch.Size([2, 64, 256]) (N=64 tokens, D=256)


## 4 · Multi-Head Self-Attention & Transformer Blocks


In [4]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim=256, num_heads=8, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        norm_x = self.norm1(x)
        attn_out, _ = self.attn(norm_x, norm_x, norm_x)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x


## 5 · Architecture 1: Vision Transformer Autoencoder (ViT-AE)


In [5]:
class VisionTransformerAutoencoder(nn.Module):
    def __init__(
        self,
        img_size=128,
        patch_size=16,
        in_channels=3,
        embed_dim=256,
        depth=6,
        num_heads=8,
        decoder_embed_dim=128,
        decoder_depth=4,
        decoder_num_heads=4,
        mlp_ratio=4.0,
        dropout=0.1
    ):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.in_channels = in_channels
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        num_patches = self.patch_embed.num_patches

        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.encoder_blocks = nn.ModuleList([
            TransformerEncoderBlock(embed_dim, num_heads, mlp_ratio, dropout)
            for _ in range(depth)
        ])
        self.encoder_norm = nn.LayerNorm(embed_dim)

        self.decoder_proj = nn.Linear(embed_dim, decoder_embed_dim)
        self.decoder_pos_embed = nn.Parameter(torch.zeros(1, num_patches, decoder_embed_dim))
        nn.init.trunc_normal_(self.decoder_pos_embed, std=0.02)

        self.decoder_blocks = nn.ModuleList([
            TransformerEncoderBlock(decoder_embed_dim, decoder_num_heads, mlp_ratio, dropout)
            for _ in range(decoder_depth)
        ])
        self.decoder_norm = nn.LayerNorm(decoder_embed_dim)
        self.decoder_pred = nn.Linear(decoder_embed_dim, patch_size * patch_size * in_channels)

    def unpatchify(self, patch_preds):
        p = self.patch_size
        h = w = self.img_size // p
        B = patch_preds.shape[0]
        x = patch_preds.reshape(shape=(B, h, w, p, p, self.in_channels))
        x = torch.einsum('nhwpqc->nchpwq', x)
        imgs = x.reshape(shape=(B, self.in_channels, h * p, w * p))
        return imgs

    def forward(self, x):
        tokens = self.patch_embed(x) + self.pos_embed
        for block in self.encoder_blocks:
            tokens = block(tokens)
        latent_tokens = self.encoder_norm(tokens)

        dec_tokens = self.decoder_proj(latent_tokens) + self.decoder_pos_embed
        for block in self.decoder_blocks:
            dec_tokens = block(dec_tokens)
        dec_tokens = self.decoder_norm(dec_tokens)

        patch_preds = self.decoder_pred(dec_tokens)
        recon = torch.sigmoid(self.unpatchify(patch_preds))
        return recon, latent_tokens

vit_ae = VisionTransformerAutoencoder(
    img_size=IMG_SIZE,
    patch_size=PATCH_SIZE,
    embed_dim=EMBED_DIM,
    depth=ENC_DEPTH,
    num_heads=NUM_HEADS,
    decoder_embed_dim=DEC_EMBED,
    decoder_depth=DEC_DEPTH,
    decoder_num_heads=4
).to(DEVICE)

total_params = sum(p.numel() for p in vit_ae.parameters())
print(f'ViT-AE Total Parameters: {total_params:,}')


ViT-AE Total Parameters: 5,885,824


## 6 · Architecture 2: Multi-Modal Construction Progress Simulation Transformer


In [6]:
class CrossAttentionBlock(nn.Module):
    def __init__(self, embed_dim=256, num_heads=8, dropout=0.1):
        super().__init__()
        self.norm_self = nn.LayerNorm(embed_dim)
        self.self_attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm_q = nn.LayerNorm(embed_dim)
        self.norm_kv = nn.LayerNorm(embed_dim)
        self.cross_attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm_mlp = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 4, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x, context):
        x = x + self.self_attn(self.norm_self(x), self.norm_self(x), self.norm_self(x))[0]
        q = self.norm_q(x)
        kv = self.norm_kv(context)
        x = x + self.cross_attn(q, kv, kv)[0]
        x = x + self.mlp(self.norm_mlp(x))
        return x


class ConstructionProgressTransformer(nn.Module):
    def __init__(
        self,
        img_size=128,
        patch_size=16,
        in_channels=3,
        num_stages=6,
        embed_dim=256,
        depth=4,
        num_heads=8
    ):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.in_channels = in_channels
        
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        num_patches = self.patch_embed.num_patches
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.stage_embedding = nn.Embedding(num_stages, embed_dim)
        self.time_mlp = nn.Sequential(
            nn.Linear(1, embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, embed_dim)
        )

        self.sim_blocks = nn.ModuleList([
            CrossAttentionBlock(embed_dim, num_heads)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)

        self.future_patch_pred = nn.Linear(embed_dim, patch_size * patch_size * in_channels)
        self.risk_patch_pred = nn.Linear(embed_dim, patch_size * patch_size * 1)

    def unpatchify(self, patch_preds, out_channels):
        p = self.patch_size
        h = w = self.img_size // p
        B = patch_preds.shape[0]
        x = patch_preds.reshape(shape=(B, h, w, p, p, out_channels))
        x = torch.einsum('nhwpqc->nchpwq', x)
        return x.reshape(shape=(B, out_channels, h * p, w * p))

    def forward(self, x_current, stage_idx, delta_days):
        vis_tokens = self.patch_embed(x_current) + self.pos_embed
        stage_token = self.stage_embedding(stage_idx).unsqueeze(1)
        time_token = self.time_mlp(delta_days).unsqueeze(1)
        context = torch.cat([stage_token, time_token], dim=1)

        tokens = vis_tokens
        for block in self.sim_blocks:
            tokens = block(tokens, context)
        tokens = self.norm(tokens)

        future_patches = self.future_patch_pred(tokens)
        risk_patches = self.risk_patch_pred(tokens)

        x_future_sim = torch.sigmoid(self.unpatchify(future_patches, self.in_channels))
        risk_map = torch.sigmoid(self.unpatchify(risk_patches, 1))
        return x_future_sim, risk_map

progress_transformer = ConstructionProgressTransformer(
    img_size=IMG_SIZE,
    patch_size=PATCH_SIZE,
    num_stages=6,
    embed_dim=EMBED_DIM,
    depth=4,
    num_heads=NUM_HEADS
).to(DEVICE)

p_params = sum(p.numel() for p in progress_transformer.parameters())
print(f'Progress Transformer Total Parameters: {p_params:,}')


Progress Transformer Total Parameters: 4,760,576


## 7 · Tensor Shape Contracts & Verification


In [7]:
test_input = torch.randn(4, 3, 128, 128).to(DEVICE)
recon, latent_tokens = vit_ae(test_input)

print('=== ViT-AE Shape Verification ===')
print(f'Input Image Shape       : {tuple(test_input.shape)}')
print(f'Latent Tokens Shape     : {tuple(latent_tokens.shape)} ([B, N_patches, D_dim])')
print(f'Reconstructed Image     : {tuple(recon.shape)}')
assert recon.shape == test_input.shape, 'Reconstruction shape mismatch!'

dummy_stage = torch.tensor([1, 2, 3, 4]).to(DEVICE)
dummy_delta = torch.tensor([[15.0], [30.0], [60.0], [90.0]]).to(DEVICE)

x_future, risk_map = progress_transformer(test_input, dummy_stage, dummy_delta)

print('\n=== Construction Progress Transformer Shape Verification ===')
print(f'Current Site State      : {tuple(test_input.shape)}')
print(f'Conditioning Stage ID   : {tuple(dummy_stage.shape)}')
print(f'Conditioning Delta Days : {tuple(dummy_delta.shape)}')
print(f'Simulated Future State  : {tuple(x_future.shape)}')
print(f'Predicted Risk Map      : {tuple(risk_map.shape)}')
assert x_future.shape == test_input.shape, 'Simulated image shape mismatch!'
assert risk_map.shape == (4, 1, 128, 128), 'Risk map shape mismatch!'
print('\nAll tensor shape contracts validated successfully!')


=== ViT-AE Shape Verification ===
Input Image Shape       : (4, 3, 128, 128)
Latent Tokens Shape     : (4, 64, 256) ([B, N_patches, D_dim])
Reconstructed Image     : (4, 3, 128, 128)

=== Construction Progress Transformer Shape Verification ===
Current Site State      : (4, 3, 128, 128)
Conditioning Stage ID   : (4,)
Conditioning Delta Days : (4, 1)
Simulated Future State  : (4, 3, 128, 128)
Predicted Risk Map      : (4, 1, 128, 128)

All tensor shape contracts validated successfully!


## 8 · Training & Loss Pipeline


In [8]:
criterion = nn.MSELoss()
optimizer = optim.AdamW(vit_ae.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

print('Training pipeline configured successfully.')


Training pipeline configured successfully.


## 9 · Architecture Comparison: AE vs. VAE vs. Transformer

| Feature / Metric | Convolutional AE | Convolutional VAE | Vision Transformer (ViT-AE / Progress TransSim) |
| :--- | :--- | :--- | :--- |
| **Spatial Modeling** | Local Receptive Fields (CNN) | Local Receptive Fields (CNN) | **Global Attention across all visual patches** |
| **Bottleneck Structure** | Deterministic 1D vector (256-D) | Gaussian distribution (mu, logvar) | **Spatial Patch Token Sequence (64 x 256)** |
| **Multi-Modal Guidance** | None (Unconditional) | None (Unconditional) | **Cross-Attention on BIM stages & timeline offsets** |
| **Generative Capability** | Reconstruction only | Random Gaussian prior sampling | **Targeted Progress Simulation & Risk Forecasting** |
| **Digital Twin Role** | Baseline feature extractor | Continuous state interpolator | **Full 4D Multi-Modal Digital Twin Simulator** |

---
